## Out of vocabulary setup 


Setting file paths

In [3]:
import os, glob, re
from collections import Counter
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer

os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")

CLEAN = "data/interim/clean"
CLEAN_DAPT = "data/interim/clean_dapt"
OUT = "data/processed"
os.makedirs(OUT, exist_ok=True)

# OOV analysis runs on the FULL DAPT corpus (both folders), since that's
# what the vocabulary needs to cover.
FILES = sorted(glob.glob(f"{CLEAN}/*.parquet") + glob.glob(f"{CLEAN_DAPT}/*.parquet"))
print(len(FILES), "files")

SAMPLE = 300_000   # texts to sample for frequency counting (speed vs coverage)
MIN_FREQ = 20      # ignore words appearing fewer than this many times

38 files


## Building vocabulary frequency table

In [4]:
# Sample texts across all subreddits proportionally, count word frequencies.
WORD = re.compile(r"[a-z][a-z'\-]+")   # lowercase alpha words, keeps don't / bias-wrecker

counts = Counter()
per_file = max(SAMPLE // len(FILES), 500)

for p in tqdm(FILES, desc="counting"):
    d = pd.read_parquet(p, columns=["text"])
    txt = d.text.dropna()
    if len(txt) > per_file:
        txt = txt.sample(per_file, random_state=0)
    for t in txt:
        counts.update(WORD.findall(t.lower()))
    del d

print("distinct words:", len(counts))
freq = pd.DataFrame(counts.items(), columns=["word", "freq"])
freq = freq[freq.freq >= MIN_FREQ].sort_values("freq", ascending=False)
print("words above min_freq:", len(freq))

counting: 100%|██████████| 38/38 [00:04<00:00,  7.93it/s]

distinct words: 123180
words above min_freq: 15955


## 03_oov — status

**Corpus:** 38 files (all subs + popculture) · 15,955 words above min_freq=20

**Results:**
- Clean tokenisation (≤2 pieces): base __% | twitter __%
- Tokens covering 85% OOV mass: __

**Curation:** KEEP = slang/ritual/harm terms · DROP = names, typos, usernames
- [ ] curated `oov_candidates.csv`
- [ ] ran cell 6 → `new_tokens.csv` (__ tokens)

**Out:** `new_tokens.csv` → feeds 04_dapt

In [5]:
# A word is "OOV-ish" if the tokenizer shatters it into many sub-pieces.
# We measure against xlm-roberta-base (your DAPT base) AND twitter-xlm-roberta
# (social-media-adapted) to see which fandom terms each already covers.

tok_base = AutoTokenizer.from_pretrained("xlm-roberta-base")
tok_tw   = AutoTokenizer.from_pretrained("cardiffnlp/twitter-xlm-roberta-base")

def frag(tok, w):
    # leading space so it's treated as a word start, matching real usage
    return len(tok.tokenize(" " + w))

words = freq.word.tolist()
freq["frag_base"] = [frag(tok_base, w) for w in tqdm(words, desc="xlm-r")]
freq["frag_tw"]   = [frag(tok_tw, w)   for w in tqdm(words, desc="twitter")]

# OOV "mass" = how much tokenizer waste this word causes overall:
# frequent AND heavily fragmented = high priority to add.
freq["mass_base"] = freq.freq * freq.frag_base
freq["mass_tw"]   = freq.freq * freq.frag_tw

freq.to_parquet(f"{OUT}/word_freq_frag.parquet")

twitter: 100%|██████████| 15955/15955 [00:00<00:00, 44401.85it/s]


tokeniser comparision

In [6]:
# How much does the social-media model already cover vs the base?
covered_base = (freq.frag_base <= 2).mean()
covered_tw   = (freq.frag_tw <= 2).mean()
print(f"words tokenised cleanly (<=2 pieces):")
print(f"  xlm-roberta-base      : {covered_base:.1%}")
print(f"  twitter-xlm-roberta   : {covered_tw:.1%}")

# words the Twitter model handles but the base doesn't -> social pretraining helps
tw_helps = freq[(freq.frag_base >= 3) & (freq.frag_tw <= 2)]
print(f"\nterms Twitter model covers that base fragments: {len(tw_helps)}")
print(tw_helps.head(20)[["word","freq","frag_base","frag_tw"]].to_string(index=False))

words tokenised cleanly (<=2 pieces):
  xlm-roberta-base      : 80.0%
  twitter-xlm-roberta   : 80.0%

terms Twitter model covers that base fragments: 0
Empty DataFrame
Columns: [word, freq, frag_base, frag_tw]
Index: []


In [7]:
# Rank OOV candidates by how much tokenizer waste they'd remove.
# Cumulative mass tells you where the 85% target (from your proposal) lands.
oov = freq[freq.frag_base >= 3].sort_values("mass_base", ascending=False).copy()
oov["cum_frac"] = oov.mass_base.cumsum() / oov.mass_base.sum()

cut85 = oov[oov.cum_frac <= 0.85]
print(f"{len(cut85)} tokens cover 85% of OOV mass (target from proposal)")
print(f"full OOV-ish list: {len(oov)} tokens")

# Columns you'll curate on: keep? which domain? is it slang vs a proper noun?
cut85 = cut85[["word","freq","frag_base","frag_tw","mass_base","cum_frac"]].copy()
cut85["keep"] = ""          # you fill: 1 = add, 0 = drop
cut85["domain"] = ""        # you fill: parasocial / financial / victim / general
cut85["note"] = ""          # you fill: why (esp. why dropped)
cut85.to_csv(f"{OUT}/oov_candidates.csv", index=False)
print("\nwrote oov_candidates.csv — curate by hand next")
print(cut85.head(30).to_string(index=False))

829 tokens cover 85% of OOV mass (target from proposal)
full OOV-ish list: 3195 tokens

wrote oov_candidates.csv — curate by hand next
      word  freq  frag_base  frag_tw  mass_base  cum_frac keep domain note
      it's 52486          3        3     157458  0.079529                 
       i'm 40924          3        3     122772  0.141538                 
     don't 40611          3        3     121833  0.203073                 
    that's 14485          3        3      43455  0.225021                 
      i've 14481          3        3      43443  0.246963                 
   they're 13413          3        3      40239  0.267287                 
     can't 12950          3        3      38850  0.286909                 
    didn't 11630          3        3      34890  0.304531                 
   doesn't 11134          3        3      33402  0.321402                 
    you're  8372          3        3      25116  0.334088                 
     she's  8279          3        3    

THIS IS TO FINALIZE THE FILES 

In [3]:
import pandas as pd, os

os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")
PROC = "data/processed"

# wherever your edited file actually is — adjust if it's not in data/processed/
src = f"{PROC}/oov_candidates.csv"
d = pd.read_csv(src)
d["keep"] = pd.to_numeric(d["keep"], errors="coerce")

# --- hygiene fixes ---
d.loc[d.word.str.lower() == "k-pop",     "domain"] = "general"
d.loc[d.word.str.lower() == "engenes",   "domain"] = "parasocial"
d.loc[d.word.str.lower() == "ni-ki",     "keep"]   = 0
d.loc[d.word.str.lower() == "lmfao",     "keep"]   = 0
d.loc[d.word.str.lower() == "prettiest", "keep"]   = 0

# standardise domain strings
d["domain"] = d["domain"].astype(str).str.strip()
d.loc[d.domain == "financial/parasocial", "domain"] = "financial"
d.loc[(d.keep == 1) & (d.domain == "generic"), "keep"] = 0

keep = d[d.keep == 1].copy()
print("final keep:", len(keep))
print(keep.domain.value_counts().to_string())

# blank-domain check — must be zero before proceeding
blank = keep[keep.domain.isin(["nan", ""]) | keep.domain.isna()]
print("keeps missing domain:", len(blank), blank.word.tolist())

# write outputs
d.to_csv(f"{PROC}/oov_candidates_clean.csv", index=False)
new_tokens = sorted(keep.word.astype(str).str.lower().unique())
pd.Series(new_tokens, name="token").to_csv(f"{PROC}/new_tokens.csv", index=False)
print("wrote new_tokens.csv:", len(new_tokens))

final keep: 314
domain
general       147
victim         97
parasocial     43
financial      27
keeps missing domain: 0 []
wrote new_tokens.csv: 314
